# 2-3 Device 이동 심화

강의 원문 대신 직접 작성하고 실행한 코드와 학습 메모를 정리했습니다.


In [1]:
# 검증 가능 정답 코드
# 모델 parameter와 입력·target device를 각각 읽어 최초 불일치 지점을 찾습니다.
# 오류 메시지 뒤 모든 Tensor를 무조건 CPU로 내리지 않고 실행 목표 device에 필요한 값만 이동합니다.
log = {"model": "cuda:0", "inputs": "cuda:0", "labels": "cpu", "offset": "cpu"}
model_device = log["model"]
# 실제 연산에 참여하며 모델과 다른 위치에 있는 항목만 이동 대상으로 잡습니다.
move = [name for name in ("inputs", "labels", "offset") if log[name] != model_device]
print("move_to_model_device:", move)
print("keep:", [name for name in ("model", "inputs") if log[name] == model_device])

move_to_model_device: ['labels', 'offset']
keep: ['model', 'inputs']


In [2]:
# 검증 가능 정답 코드
# Tensor 두 개로 구성된 (x, y) batch를 지정 device로 옮기며, 두 반환값을 모두 새 변수에 저장합니다.
# helper 반환 뒤 원본 batch를 제자리 변경하지 않았는지도 확인해 호출자 부작용을 막습니다.
import torch
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = nn.Linear(3, 2).to(device)

def move_batch(batch, device):
    x, y = batch
    # Tensor.to는 이동된 Tensor를 반환하므로 두 반환값을 모두 저장합니다.
    return x.to(device), y.to(device)

x, y = move_batch((torch.randn(4, 3), torch.tensor([0, 1, 0, 1])), device)
model_device = next(model.parameters()).device
assert model_device == x.device == y.device
print("devices_equal:", model_device == x.device == y.device)
print("output_shape:", tuple(model(x).shape))

devices_equal: True
output_shape: (4, 2)


In [3]:
# 검증 가능 정답 코드
# device 계약을 먼저 통과한 실행만 처리량 재측정 후보로 남기고 예상값을 실측값처럼 승인하지 않습니다.
# 수정 전 수치와 수정 뒤 예정 수치를 같은 열로 비교하지 않고 재측정 상태를 명시합니다.
runs = {
    "A": {"devices": ["cpu", "cpu", "cpu", "cpu"], "throughput": 42, "headroom": 70},
    "B": {"devices": ["cuda:0", "cuda:0", "cpu", "cuda:0"], "throughput": 118, "headroom": 24},
    "C": {"devices": ["cuda:0", "cuda:0", "cuda:0", "cpu"], "throughput": 130, "headroom": 6},
}
routes = {}
for name, run in runs.items():
    devices_match = len(set(run["devices"])) == 1
    if devices_match and run["throughput"] >= 80:
        routes[name] = "즉시 실행"
    elif not devices_match and run["headroom"] >= 15 and run["throughput"] >= 80:
        mismatched_fields = [
            field for field, device in zip(("model", "input", "label", "mask"), run["devices"])
            if device != run["devices"][0]
        ]
        routes[name] = "수정 후 재측정:" + ",".join(mismatched_fields)
    else:
        reason = "throughput" if run["throughput"] < 80 else "memory_headroom"
        routes[name] = "보류:" + reason
next_candidate = next((name for name, route in routes.items() if route.startswith("수정 후")), "없음")
print("routes:", routes)
print("next_candidate:", next_candidate)

routes: {'A': '보류:throughput', 'B': '수정 후 재측정:label', 'C': '보류:memory_headroom'}
next_candidate: B
